# MonopolyZero — oracle hybrid checkpoint labels

Runs `oracle.label_gen --calibrate` (hybrid play, 128-sim Max-N labels only at buy/build/trade/auction checkpoints, self+pool-with-ASU lineup mix) for one seed range and drops `labels_seed{start}_{end}.npz` (+ matching `.json` metadata) into a shared Drive folder. For automated runs, upload `/content/monopolyzero-oracle-job.json` before executing this notebook.

In [ ]:
from pathlib import Path
import json, os, re, subprocess, sys, tarfile, urllib.request

CONTENT = Path(os.environ.get('MONOPOLYZERO_CONTENT', '/content'))
JOB = {
    'commit': '396cc70d376b622062704bdf64162d13471d4490',
    'repo': 'ToprakG/DeepRL_Monopoly',
    'games': 467,
    'seed_base': 20000,
    'max_rounds': 200,
    'drive_folder': 'MyDrive/labels',
    'mount_drive': True,
}
job_path = CONTENT / 'monopolyzero-oracle-job.json'
if job_path.exists():
    JOB.update(json.loads(job_path.read_text()))
assert re.fullmatch(r'[0-9a-f]{40}', JOB['commit'])
assert re.fullmatch(r'[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+', JOB['repo'])
assert int(JOB['games']) > 0 and int(JOB['max_rounds']) > 0
seed_start, seed_end = int(JOB['seed_base']), int(JOB['seed_base']) + int(JOB['games']) - 1
SHARD_NAME = f'labels_seed{seed_start}_{seed_end}'
OUTPUT = CONTENT / f'{SHARD_NAME}.json'  # oracle.label_gen writes OUTPUT (meta) + .npz + .jsonl siblings
STATUS_PATH = CONTENT / f'{SHARD_NAME}.status.json'
print(json.dumps({**JOB, 'shard_name': SHARD_NAME}, indent=2, sort_keys=True))

In [ ]:
ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
STATUS = {**JOB, 'shard_name': SHARD_NAME, 'state': 'running', 'cpu_count': os.cpu_count(), 'ram_gib': round(ram_gib, 2)}
STATUS_PATH.write_text(json.dumps(STATUS, indent=2, sort_keys=True) + '\n')
print(f"CPU cores: {os.cpu_count()} | RAM: {ram_gib:.2f} GiB")

In [ ]:
archive_path = CONTENT / f"DeepRL_Monopoly-{JOB['commit']}.tar.gz"
urllib.request.urlretrieve(
    f"https://codeload.github.com/{JOB['repo']}/tar.gz/{JOB['commit']}",
    archive_path,
)
with tarfile.open(archive_path, 'r:gz') as archive:
    members = archive.getmembers()
    roots = {Path(member.name).parts[0] for member in members if member.name}
    assert len(roots) == 1
    assert not any(member.name.startswith('/') or '..' in Path(member.name).parts for member in members)
    archive.extractall(CONTENT, filter='data')
REPOSITORY_ROOT = CONTENT / roots.pop()
assert (REPOSITORY_ROOT / 'oracle').is_dir()
print(f'Repository: {REPOSITORY_ROOT}')

In [ ]:
try:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'torch',
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'oracle.label_gen',
        '--calibrate',
        '--games', str(JOB['games']),
        '--seed', str(JOB['seed_base']),
        '--output', str(OUTPUT),
    ], cwd=REPOSITORY_ROOT, check=True)
except Exception as exc:
    STATUS.update(state='failed', error=f'{type(exc).__name__}: {exc}')
    raise
else:
    STATUS.update(state='complete', output=str(OUTPUT.with_suffix('.npz')))
finally:
    temporary = STATUS_PATH.with_suffix('.tmp')
    temporary.write_text(json.dumps(STATUS, indent=2, sort_keys=True) + '\n')
    temporary.replace(STATUS_PATH)
print(json.dumps(STATUS, indent=2, sort_keys=True))

## Sanity check the shard before uploading

In [ ]:
import numpy as np
with np.load(OUTPUT.with_suffix('.npz'), allow_pickle=False) as shard:
    print({
        'labels': len(shard['states']),
        'state_shape': shard['states'].shape,
        'mask_shape': shard['legal_masks'].shape,
        'actors': np.bincount(shard['actors'] + 1).tolist(),
    })

## Copy the shard to the shared Drive folder

Set `JOB['mount_drive'] = False` (via the uploaded job JSON) to skip this and copy the three `labels_seed*` files off `/content` manually instead.

In [ ]:
if JOB.get('mount_drive', True):
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = Path('/content/drive') / JOB['drive_folder']
    drive_dir.mkdir(parents=True, exist_ok=True)
    import shutil
    for suffix in ('.npz', '.json', '.jsonl'):
        source = OUTPUT.with_suffix(suffix)
        if source.exists():
            shutil.copy2(source, drive_dir / source.name)
    print(f'Copied {SHARD_NAME}.{{npz,json,jsonl}} -> {drive_dir}')
else:
    print('mount_drive=False; shard left at', OUTPUT.with_suffix('.npz'))